In [1]:
!git clone https://github.com/bsesic/hebrewmnist.git

Cloning into 'hebrewmnist'...
remote: Enumerating objects: 349, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (348/348), done.
remote: Total 349 (delta 1), reused 349 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (349/349), 19.09 MiB | 34.71 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import TensorDataset, random_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
'''
What needs to change before loading
Dataset	Size	Mode	Fix needed
Latin	28×28 (CSV) L	Fix rotation/flip
Arabic	32×32	RGB	    Resize + grayscale
Greek	28×28	L	    Nothing
Hindi	32×32	L	    Resize only
Korean	64×64	RGB	    Resize + grayscale
Hebrew	51×104	RGB	    Resize + grayscale
'''

# Pre Processing

In [3]:
std_transform = transforms.Compose([
    transforms.Grayscale(1),           # → grayscale
    transforms.Resize((28, 28)),       # → 28×28
    transforms.ToTensor(),             # → [0,1]
    transforms.Lambda(lambda x: x.view(-1))  # → flatten 784
])

In [4]:
train_df = pd.read_csv('/kaggle/input/datasets/crawford/emnist/emnist-byclass-train.csv', header=None)
test_df  = pd.read_csv('/kaggle/input/datasets/crawford/emnist/emnist-byclass-test.csv',  header=None)

y_train = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.long)
x_train = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0

y_test  = torch.tensor(test_df.iloc[:, 0].values, dtype=torch.long)
x_test  = torch.tensor(test_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0


def fix_emnist(x_flat):
    imgs = x_flat.reshape(-1, 28, 28)     
    imgs = torch.rot90(imgs, k=1, dims=[1,2])  
    imgs = torch.flip(imgs, dims=[2])      
    return imgs.reshape(-1, 784)           

x_train = fix_emnist(x_train)
x_test  = fix_emnist(x_test)

latin_train = TensorDataset(x_train, y_train)
latin_test  = TensorDataset(x_test,  y_test)

print(f"Latin  → train: {len(latin_train)} | test: {len(latin_test)} | classes: {y_train.unique().shape[0]}")

Latin  → train: 697932 | test: 116323 | classes: 62


**ARABIC**

In [5]:
def load_arabic(folder):
    files = sorted(f for f in os.listdir(folder) if f.endswith('.jpg'))
    imgs, raw_labels = [], []
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert('L').resize((28,28))
        imgs.append(np.array(img))
        raw_labels.append(re.sub(r'\d+', '', f.replace('.jpg', '')))
    x = torch.tensor(np.stack(imgs), dtype=torch.float32) / 255.0
    return x.reshape(-1, 784), raw_labels

x_tr, lbl_tr = load_arabic('/kaggle/input/datasets/rashwan/arabic-chars-mnist/train')
x_te, lbl_te = load_arabic('/kaggle/input/datasets/rashwan/arabic-chars-mnist/test')
le_ar        = LabelEncoder().fit(lbl_tr)
arabic_train = TensorDataset(x_tr, torch.tensor(le_ar.transform(lbl_tr), dtype=torch.long))
arabic_test  = TensorDataset(x_te, torch.tensor(le_ar.transform(lbl_te), dtype=torch.long))
print(f"Arabic  → train: {len(arabic_train)} | test: {len(arabic_test)} | classes: {len(le_ar.classes_)} ")


Arabic  → train: 13440 | test: 3360 | classes: 28 


**Greek...**

In [6]:
greek_data          = ImageFolder('/kaggle/input/datasets/sayangupta001/mnist-greek-letters/Greek_Letters', std_transform)
tr_n, te_n          = int(0.8*len(greek_data)), len(greek_data)-int(0.8*len(greek_data))
greek_train, greek_test = random_split(greek_data, [tr_n, te_n])
print(f"Greek   → train: {len(greek_train)} | test: {len(greek_test)} | classes: {len(greek_data.classes)} ")


Greek   → train: 480 | test: 120 | classes: 24 


**HINDI...**

In [7]:
hindi_train = ImageFolder('/kaggle/input/datasets/berlinsweird/devanagari/Hindi/Train', std_transform)
hindi_test  = ImageFolder('/kaggle/input/datasets/berlinsweird/devanagari/Hindi/Test',  std_transform)
print(f"Hindi   → train: {len(hindi_train)} | test: {len(hindi_test)} | classes: {len(hindi_train.classes)} ")

Hindi   → train: 78200 | test: 13800 | classes: 46 


**Korean...**

In [8]:
korean_data          = ImageFolder('/kaggle/input/datasets/jkim289/handwritten-korean-characters/Hangul Database/Hangul Database', std_transform)
tr_n, te_n           = int(0.8*len(korean_data)), len(korean_data)-int(0.8*len(korean_data))
korean_train, korean_test = random_split(korean_data, [tr_n, te_n])
print(f"Korean  → train: {len(korean_train)} | test: {len(korean_test)} | classes: {len(korean_data.classes)} ")


Korean  → train: 5120 | test: 1280 | classes: 64 


**HEBREW**

In [9]:
hebrew_data          = ImageFolder('/kaggle/working/hebrewmnist/hebrew_letters', std_transform)
tr_n, te_n           = int(0.8*len(hebrew_data)), len(hebrew_data)-int(0.8*len(hebrew_data))
hebrew_train, hebrew_test = random_split(hebrew_data, [tr_n, te_n])
print(f"Hebrew  → train: {len(hebrew_train)} | test: {len(hebrew_test)} | classes: {len(hebrew_data.classes)} ")

Hebrew  → train: 245 | test: 62 | classes: 28 


In [10]:
from torch.utils.data import DataLoader
from itertools import cycle

BATCH_SIZE = 32

train_loaders = [
    DataLoader(latin_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(arabic_train, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(greek_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(hindi_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(korean_train, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(hebrew_train, batch_size=BATCH_SIZE, shuffle=True),
]

test_loaders = [
    DataLoader(latin_test,  batch_size=256, shuffle=False),
    DataLoader(arabic_test, batch_size=256, shuffle=False),
    DataLoader(greek_test,  batch_size=256, shuffle=False),
    DataLoader(hindi_test,  batch_size=256, shuffle=False),
    DataLoader(korean_test, batch_size=256, shuffle=False),
    DataLoader(hebrew_test, batch_size=256, shuffle=False),
]

# Infinite iterators — small datasets loop back automatically
inf_iters = [cycle(loader) for loader in train_loaders]

SCRIPT_NAMES   = ["Latin","Arabic","Greek","Hindi","Korean","Hebrew"]
SCRIPT_CLASSES = [62, 28, 24, 46, 64, 28]

for i, (name, loader) in enumerate(zip(SCRIPT_NAMES, train_loaders)):
    print(f"  m={i} {name}: {len(loader)} batches per epoch")

  m=0 Latin: 21811 batches per epoch
  m=1 Arabic: 420 batches per epoch
  m=2 Greek: 15 batches per epoch
  m=3 Hindi: 2444 batches per epoch
  m=4 Korean: 160 batches per epoch
  m=5 Hebrew: 8 batches per epoch
